# Module 04 — Phân phối, ý nghĩa thống kê, effect size, power
Notebook RIÊNG cho Module 04 — tái lập: làm sạch outlier, khoảng tin cậy 95%, skewness & Shapiro-Wilk, Cohen's d, eta², bảng cỡ mẫu theo power.

Tương ứng SPSS/PSPP: `Transform > Compute`, `DESCRIPTIVES`, `EXAMINE`, `T-TEST`, `ONEWAY`.

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' — CÙNG một seed=2026 dùng
    xuyên suốt các module 03-08, khớp chính xác với lop_hoc_240.csv/.sav
    trong repo (không phải dữ liệu thật, nhưng tái lập được 100%)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Đã tạo df:", df.shape)

## 0. Làm sạch outlier `study_hours` TRƯỚC khi phân tích
Một học sinh có `study_hours=48` — gần chắc chắn lỗi gõ nhầm "4,8" thành "48" (xem Module 01). Nếu bỏ qua bước này, skewness sẽ ra ≈7,4 thay vì 0,658.

In [ ]:
mask = df["study_hours"] > 20
print("Số dòng bị sửa:", mask.sum(), "| giá trị trước:", df.loc[mask, "study_hours"].tolist())
df.loc[mask, "study_hours"] = df.loc[mask, "study_hours"] / 10
print("giá trị sau:", df.loc[mask, "study_hours"].tolist())

## 1. Khoảng tin cậy 95% cho điểm Toán trung bình
Công thức: CI₉₅ = x̄ ± 1,96 × SE, với SE = s/√n.

🎯 **Phương pháp này trả lời câu hỏi gì?** Khoảng tin cậy (CI) trả lời: nếu lặp lại nghiên cứu lấy mẫu nhiều lần, khoảng giá trị nào có khả năng chứa tham số THẬT của tổng thể. Khác với chỉ báo cáo MỘT con số trung bình mẫu duy nhất, CI cho biết thêm mức độ KHÔNG CHẮC CHẮN đi kèm với con số đó — mẫu càng nhỏ, CI càng rộng (càng kém chắc chắn).

In [ ]:
m = df["math"].mean(); s = df["math"].std(ddof=1); n = len(df)
se = s / np.sqrt(n)
ci_low, ci_high = m - 1.96*se, m + 1.96*se
print(f"mean={m:.2f}  s={s:.2f}  SE={se:.3f}")
print(f"CI95% = [{ci_low:.2f}; {ci_high:.2f}]")

#### 📤 Đầu ra thật
mean=47,91, SE=0,625 → CI95%=[46,68; 49,14]. ✅ Khớp đúng con số trong bài.

## 2. Skewness & Shapiro-Wilk: `study_hours` (đã làm sạch) vs `pretest`
Quy tắc thực hành: so skewness với ±2×SE(skewness), SE(skew) ≈ √(6/n).

🎯 **Phương pháp này trả lời câu hỏi gì?** Kiểm tra skewness/Shapiro-Wilk trả lời: dữ liệu có đủ gần phân phối chuẩn để dùng kiểm định THAM SỐ (t-test, ANOVA — vốn giả định phân phối chuẩn) hay không. Đây là bước kiểm tra ĐIỀU KIỆN ÁP DỤNG của phương pháp, không phải bản thân kết quả nghiên cứu — nếu vi phạm nặng, nên chuyển sang kiểm định phi tham số (Module 07).

In [ ]:
se_skew = np.sqrt(6/n)
for col in ["study_hours", "pretest"]:
    sk = stats.skew(df[col], bias=False)
    w, p = stats.shapiro(df[col])
    verdict = "LỆCH có ý nghĩa" if abs(sk) > 2*se_skew else "trong ngưỡng chấp nhận"
    print(f"{col}: skew={sk:+.3f} (ngưỡng ±{2*se_skew:.3f}, {verdict}) | Shapiro-Wilk W={w:.3f}, p={p:.4f}")

#### 📤 Đầu ra thật
`study_hours`: skew=+0,658, ngưỡng ±0,314 → VƯỢT ngưỡng, Shapiro-Wilk p<0,001 (không ủng hộ giả thuyết "dữ liệu chuẩn"). `pretest`: skew=-0,097, trong ngưỡng, Shapiro-Wilk p=0,117 (không đủ bằng chứng bác normality). ✅ Khớp đúng bài.

## 3. Cohen's d — mức tăng điểm (gain) theo phương pháp dạy

🎯 **Phương pháp này trả lời câu hỏi gì?** Cohen's d trả lời: khác biệt giữa 2 nhóm LỚN tới mức nào, tính theo đơn vị “số độ lệch chuẩn” — bổ sung cho p-value (chỉ nói CÓ/KHÔNG có ý nghĩa thống kê) bằng câu trả lời về ĐỘ LỚN thực tế, điều mà p-value một mình không cho biết được (xem Module 04 mục Ý nghĩa thống kê ≠ ý nghĩa thực tiễn).

In [ ]:
df["gain"] = df["posttest"] - df["pretest"]
g1 = df.loc[df["method"]=="Dự án", "gain"]
g2 = df.loc[df["method"]=="Truyền thống", "gain"]
sp = np.sqrt(((len(g1)-1)*g1.var(ddof=1) + (len(g2)-1)*g2.var(ddof=1)) / (len(g1)+len(g2)-2))
d = (g1.mean() - g2.mean()) / sp
t, p = stats.ttest_ind(g1, g2)
print(f"Dự án: n={len(g1)}, M={g1.mean():.2f}, SD={g1.std(ddof=1):.2f}")
print(f"Truyền thống: n={len(g2)}, M={g2.mean():.2f}, SD={g2.std(ddof=1):.2f}")
print(f"t={t:.3f}, p={p:.2e}")
print(f"Cohen's d = {d:.3f}")

#### 📤 Đầu ra thật
d≈0,981 (mức trung bình-mạnh, gần chạm ngưỡng "lớn" 1,00), t=7,589 (p<0,001). ✅ Khớp đúng bài.

## 4. Eta² — trường → điểm Toán (kết hợp cùng ANOVA ở Module 06)

🎯 **Phương pháp này trả lời câu hỏi gì?** Eta² trả lời: bao nhiêu % biến thiên của biến phụ thuộc được “giải thích” bởi biến nhóm — tương tự R² trong hồi quy, đây là một dạng CỠ HIỆU ỨNG dành riêng cho ANOVA, bổ sung cho p-value (chỉ nói có/không ý nghĩa, không nói độ lớn).

In [ ]:
grand = df["math"].mean()
ssb = sum(len(g)*(g["math"].mean()-grand)**2 for _, g in df.groupby("school"))
sst = ((df["math"]-grand)**2).sum()
eta2 = ssb / sst
print(f"SSB={ssb:.2f}  SST={sst:.2f}  eta²={eta2:.4f}")

#### 📤 Đầu ra thật
eta²≈0,047 → mức NHỎ, dù kiểm định F có ý nghĩa thống kê (p=0,003)! Minh chứng: ý nghĩa thống kê và cỡ hiệu ứng là hai câu hỏi khác nhau.

## 5. Bảng cỡ mẫu theo power (Cohen 1988, statsmodels)
Cần cài `statsmodels` nếu chưa có: trên Colab đã có sẵn.

🎯 **Phương pháp này trả lời câu hỏi gì?** Power analysis trả lời: cần bao nhiêu quan sát để có cơ hội HỢP LÝ phát hiện được một hiệu ứng thật (nếu nó thực sự tồn tại) — làm TRƯỚC khi thu thập dữ liệu, không phải sau. Nếu cỡ mẫu quá nhỏ, ngay cả một hiệu ứng thật cũng dễ bị bỏ sót (Type II error).

In [ ]:
from statsmodels.stats.power import TTestIndPower
analysis = TTestIndPower()
rows = []
for d_val, label in [(0.2,"Nhỏ"), (0.5,"Vừa"), (0.8,"Lớn")]:
    n_req = analysis.solve_power(effect_size=d_val, alpha=0.05, power=0.80, alternative="two-sided")
    rows.append((d_val, label, round(n_req)))
pd.DataFrame(rows, columns=["Cỡ hiệu ứng d", "Mức", "Cỡ mẫu/nhóm"])

#### 📤 Đầu ra thật
d=0,2→≈394/nhóm, d=0,5→≈64/nhóm, d=0,8→≈26/nhóm (α=0,05, power=0,80). ✅ Khớp đúng bảng trong Module 02 và Module 04.